### Personal Environmental Impact Tracker
i am building a Personal Environmental Impact Tracker that simulates my 23 years of travel history to quantify and analyse the environmental impact of personal travel habits. Since real-world data such as Google Maps Takeout was unavailable, i generated synthetic travel data representing my movement patterns across my different age brackets. Each trip includes mode of transport, origin, destination, distance, time, and carbon emissions. The goal is to use this data to train a model that can suggest more sustainable transportation alternatives (like switching from car to bike, public transport, or walking) based on trip context (distance, duration, purpose). Ultimately, this model integrates into a dashboard that visualises insights, lets users input their own data, and encourages behavior change through personalised recommendations and goal tracking.

In [3]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import joblib

In [4]:
df = pd.read_csv("travel_data.csv") 

In [5]:
# mapping 'Purpose'
# doing this to add semantic meaning to trips which is gonna help the model genralise recommendations b etter
purpose_map = {
    'Work': 'Commute', 'Part-time Work': 'Commute', 'University': 'Commute', 'Gym': 'Commute',
    'School': 'Education',

    'Grocery Store': 'Errand', 'Shopping Centre': 'Errand', 'Library': 'Errand', 'Pharmacy': 'Errand',

    'Hospital': 'Health',

    'Cricket Ground': 'Leisure', 'Museum': 'Leisure', 'Park': 'Leisure',
    "Friend's Place": 'Leisure', 'Stadium': 'Leisure', 'Cinema': 'Leisure',
    'Cafe': 'Leisure', 'Beach': 'Leisure',

    'Train Station': 'Transit', 'Airport': 'Travel',

    "Grandparents' House": 'Family'
}

df['Purpose'] = df['To'].map(purpose_map).fillna("Other")

In [6]:
# carbon emission factors (grams per km) for each mode 
co2_per_km = {
    "Walk": 0,
    "Bicycle": 0,
    "Car": 192,
    "Bus": 105,
    "Train": 41,
    "Tram": 60,
    "Flight": 150
}

# rec logic - considers both distance and current emissions:
    # walk for very short trips (≤2 km)
    # bicycle for short trips (≤5 km)
    # pt. for mid-range (≤10 km)
# BUT it only recommends a change if the greener mode results in at least 30% carbon savings
def recommend_mode(row):
    # calculating current trip's emissions
    current_emission = row['Distance_km'] * co2_per_km.get(row['Mode'], 100)

    # greener alternative based on dist and purpose
    if row['Distance_km'] <= 2:
        alt_mode = 'Walk'
    elif row['Distance_km'] <= 5:
        alt_mode = 'Bicycle'
    elif row['Distance_km'] <= 10 and row['Purpose'] in ['Commute', 'Errand']:
        alt_mode = 'Public Transport'
    else:
        # for longer trips or non-commute/errand purposes, no change is suggested
        return 'Same'

    # calculating what the emissions would be if the alternative mode were used
    alt_emission = row['Distance_km'] * co2_per_km.get(alt_mode, 100)

    # recommend the new mode only if it saves at least 30% emissions
    if alt_emission < current_emission * 0.7:
        return alt_mode
    else:
        return 'Same'

df['Recommended_Mode'] = df.apply(recommend_mode, axis=1)

In [7]:
# filtering out 'Same' — only want learnable changes
# removes those trips where no change is recommended (tllhe current mode is already ideal)
df = df[df['Recommended_Mode'] != 'Same']

In [8]:
X = df[['Distance_km', 'Duration_min', 'Purpose']] # features to be used for training the model
X = pd.get_dummies(X)  # one-hot encoding 'Purpose'
y = df['Recommended_Mode'] # target var.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)
# 'stratify=y' to ensure each mode of transport class is represented proportionally in both sets

In [10]:
# using decision tree cause:
    # the recommendation logic here is a set of if-then rules
    # need to handle both numerical and categorical (Purpose) features
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=4, random_state=42)

In [11]:
joblib.dump(model, "recommendation_model.pkl")
joblib.dump(X.columns.tolist(), "model_features.pkl")

['model_features.pkl']

In [12]:
print("Model and feature names saved.")

Model and feature names saved.
